# Get Dark Area Values from Across SSA and Mini-Grid Brightness Values
This notebook takes an open source-data set of mini-grid locations across Sub-Saharan Africa and extracts the nightlight brightness values from the VIIRS nightlights data set for each mini-grid location. It also extracts the nightlight brightness values for a set of random dark areas across SSA to use as a comparison, sampled specifically to be in areas with low- to no-inhabitants as baselines for nighttime brightness.
![africa-night-lights](figures/website/africa-night-lights.png)

In [ ]:
# import geemap and geopandas for spatial data handling and their dependencies
import ee
import geemap
import pandas as pd
import geopandas as gpd

## Authenticate & Initialize GEE

Requires a [Google Cloud Project](https://console.cloud.google.com/projectcreate) and to enable the [Earth Engine API](https://console.cloud.google.com/apis/api/earthengine.googleapis.com) for the project. Find detailed instructions [here](https://book.geemap.org/chapters/01_introduction.html#earth-engine-authentication).

In [20]:
ee.Initialize()

## Create a GEEMap Object

In [21]:
m = geemap.Map(
    center=[-5, 15], 
    zoom=3, 
    basemap = 'Esri.WorldImagery',
    height = 1000
)

## Add Layers to the Map
Let's visualize the VIIRS nighttime lights on the map to help guide our sampling of dark areas.

In [22]:
# add nightlights median
# https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMSLCFG
dataset_night = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG') \
                  .filter(ee.Filter.date('2020-01-01', '2024-01-01'))
nighttime = dataset_night.select('avg_rad')
image_night = nighttime.median()
nighttimeVis = {'min': 0.0, 'max': 2.0}
m.addLayer(image_night, nighttimeVis, 'Nighttime', True)


In [ ]:
# add latest world pop data layer
# https://developers.google.com/earth-engine/datasets/catalog/WorldPop_GP_100m_pop
# dataset_pop = ee.ImageCollection('WorldPop/GP/100m/pop') \
#                   .filter(ee.Filter.date('2020-01-01', '2024-01-01'))
# pop = dataset_pop.select('population')
# image_pop = pop.median()
# popVis = {'min': 0.0, 'max': 20.0, 'palette': ['24126c', '1fff4f', 'd4ff50']}
# m.addLayer(image_pop, popVis, 'Population', True)

In [24]:
# add DarkMatter country labels to basemap
m.add_basemap('CartoDB.VoyagerOnlyLabels')

In [25]:
# show map
m

Map(center=[-5, 15], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tran…

In [26]:
# set zoom to 4.5
m.setCenter(10, -1, 4.5)

## Draw, Save, then Read in Ocean, Desert, and Jungle Features
Looking at the map above, we can then hand-draw polygons around areas that are clearly ocean, desert, or jungle. These areas should have very low to no inhabitants, and thus should have very low nightlight brightness values. After drawing the polygons, we can save them as a GeoJSON file and then read them back in as a GeoDataFrame for sampling.

In [27]:
# save the ocean polygon drawn on the map to a ee.Feature object
# ocean_feat = m.draw_last_feature
# # export the ee.Feature object to a geojson
# geemap.ee_to_geojson(ocean_feat, 'data/dark_africa/ocean2.geo.json')


In [28]:
# save desert polygon
# desert_feat = m.draw_last_feature
# geemap.ee_to_geojson(desert_feat, 'data/dark_africa/desert.geo.json')

In [29]:
# jungle poly
# jungle_feat = m.draw_last_feature
# geemap.ee_to_geojson(jungle_feat, 'data/dark_africa/jungle.geo.json')

In [30]:
# read back in geojson files to ee features
ocean_feat = geemap.geojson_to_ee('data/dark_africa/ocean2.geo.json')
desert_feat = geemap.geojson_to_ee('data/dark_africa/desert.geo.json')
jungle_feat = geemap.geojson_to_ee('data/dark_africa/jungle.geo.json')

In [31]:
# add the features to the map
m.addLayer(ocean_feat, {'color': 'blue'}, 'ocean')
m.addLayer(desert_feat, {'color': 'orange'}, 'desert')
m.addLayer(jungle_feat, {'color': 'darkgreen'}, 'jungle')

## Get Countries with Mini-Grids and Polygons
Now that we have our "dark areas" defined, we can read in the open-source mini-grid data sets and get the countries that have mini-grids in them. We'll skip the Cross-Boundary and PowerGen data sets for now since their mini-grid locations are private data.

In [32]:
# get list of countries in CLUB-ER dataset
cluber_df = pd.read_csv('data/cluber/cluber_sites_clean.csv')
# cbil_df = pd.read_csv('data/cbil/site_data.csv')
# pg_df = pd.read_csv('data/pg/site_data.csv')

# covert date_commissioned to date
cluber_df['date_commissioned'] = pd.to_datetime(cluber_df['date_commissioned'])
# filter out sites commissioned before 2014-01-01
cluber_df = cluber_df[cluber_df['date_commissioned'] >= '2014-01-01']

# get a unique list of countries
cluber_countries = cluber_df['country'].unique()
print('Club-ER Countries: ', cluber_countries)
# cbil_countries = cbil_df['country'].unique()
# print('Cross Boundary Countries: ', cbil_countries)
# pg_countries = pg_df['country'].unique()
# print('PowerGen Countries: ', pg_countries)

# count the number of mini-grids in each country
cluber_count = cluber_df['country'].value_counts()
# cbil_count = cbil_df['country'].value_counts()
# pg_count = pg_df['country'].value_counts()
# sum together by country
# all_minigrid_counts = cluber_count.add(cbil_count, fill_value=0).add(pg_count, fill_value=0).sort_index()
all_minigrid_counts = cluber_count
# conver to integers
all_minigrid_counts = all_minigrid_counts.astype(int)
print('All Mini-Grid Counts: ', all_minigrid_counts)



# modify list of countries
# combine the three lists of countries into one with just unique values
# all_minigrid_countries = list(set(cluber_countries) | set(cbil_countries) | set(pg_countries))
all_minigrid_countries = list(set(cluber_countries))
# remove nan values
all_minigrid_countries = [x for x in all_minigrid_countries if str(x) != 'nan']
# exclude Haiti
all_minigrid_countries = [x for x in all_minigrid_countries if x != 'Haiti']
# rename DR Congo to Democratic Republic of the Congo
all_minigrid_countries = [x if x != 'DR Congo' else 'Democratic Republic of the Congo' for x in all_minigrid_countries]
# rename Tanzania to United Republic of Tanzania
all_minigrid_countries = [x if x != 'Tanzania' else 'United Republic of Tanzania' for x in all_minigrid_countries]

print('All Countries', all_minigrid_countries)
print('Number of countries: ', len(all_minigrid_countries))
# export the list of countries to a csv
pd.DataFrame(all_minigrid_countries, columns=['country']).to_csv('data/dark_africa/countries.csv', index=False)

Club-ER Countries:  ['Angola' 'Burkina Faso' 'Cameroon' 'DR Congo' 'Ethiopia' 'Ghana' 'Kenya'
 'Liberia' 'Madagascar' 'Mali' 'Mauritania' 'Senegal' 'Tanzania' 'Togo'
 'Zambia' 'Zimbabwe']
All Mini-Grid Counts:  country
Mali            187
Togo            107
DR Congo        101
Senegal          90
Kenya            88
Tanzania         51
Cameroon         27
Mauritania       20
Madagascar       16
Burkina Faso     15
Liberia          11
Ghana             2
Angola            1
Ethiopia          1
Zambia            1
Zimbabwe          1
Name: count, dtype: int64
All Countries ['Democratic Republic of the Congo', 'Angola', 'Ghana', 'Kenya', 'Madagascar', 'Cameroon', 'Mali', 'Ethiopia', 'Mauritania', 'Zimbabwe', 'Burkina Faso', 'Liberia', 'United Republic of Tanzania', 'Zambia', 'Senegal', 'Togo']
Number of countries:  16


In [ ]:
# overlay country boundaries with white borders on the map
countries = ee.FeatureCollection('FAO/GAUL/2015/level0')
style = {'color': 'ffffffff', 'width': 2, 'lineType': 'solid', 'opacity': 1}
# m.addLayer(countries, style, 'Countries', False)

In [34]:

# create a fc of just the countries in all_minigrid_countries
all_minigrid_countries_fc = countries.filter(ee.Filter.inList('ADM0_NAME', all_minigrid_countries))
m.addLayer(all_minigrid_countries_fc, style, 'Countries', True)

# note: this isn't styling the countries correctly
# the "fillColor" parameter doesn't seem to work

# count the number of countries in all_minigrid_countries_fc
print('countries included in the map: ', all_minigrid_countries_fc.aggregate_array('ADM0_NAME').getInfo())
print('Number of countries:')
all_minigrid_countries_fc.size()

countries included in the map:  ['Zambia', 'Kenya', 'Madagascar', 'United Republic of Tanzania', 'Zimbabwe', 'Ethiopia', 'Democratic Republic of the Congo', 'Cameroon', 'Angola', 'Togo', 'Senegal', 'Mauritania', 'Mali', 'Liberia', 'Ghana', 'Burkina Faso']
Number of countries:


In [35]:
# try to fuse the geometries of the 20 countries
all_minigrid_countries_fused_fc = ee.FeatureCollection(all_minigrid_countries_fc.union())

# add to map in gray
# m.addLayer(all_minigrid_countries_fused_fc, {'color': 'gray'}, 'Fused Countries', True)

In [36]:
# add club-er sites to the map
cluber_fc = geemap.geojson_to_ee('data/cluber/cluber_sites.geojson')
m.addLayer(cluber_fc, {'color': 'purple'}, 'Club-ER Sites')

## Get the Landcover Data from Google Earth Engine
Now, we want to sample populated areas as well as the dark areas. To do this, we can use the ESA WorldCover landcover data set from GEE to identify areas that are classified as "urban". We will also find areas that are at least 10 kilometers away from any urban area to define "rural" areas.

In [37]:
# pull in a global high resolution land cover dataset
# https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200
landcover = ee.ImageCollection('ESA/WorldCover/v200').first()

landcover_africa = landcover.clip(all_minigrid_countries_fc)

visualization = {
  'bands': ['Map'],
}

print(landcover_africa.select('Map').getInfo())

# m.addLayer(landcover_africa, visualization, 'Landcover', False)

# # inspect this image
# print(landcover_africa.getInfo())
# # inspect the bands of landcover_africa
# print(landcover_africa.bandNames().getInfo())
# # inspect the values of the band 'Map'

{'type': 'Image', 'bands': [{'id': 'Map', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [4320000, 1728000], 'crs': 'EPSG:4326', 'crs_transform': [8.333333333333333e-05, 0, -180, 0, -8.333333333333333e-05, 84]}], 'version': 1746488833928724, 'id': 'ESA/WorldCover/v200/2021', 'properties': {'Map_class_names': ['Tree cover', 'Shrubland', 'Grassland', 'Cropland', 'Built-up', 'Bare / sparse vegetation', 'Snow and ice', 'Permanent water bodies', 'Herbaceous wetland', 'Mangroves', 'Moss and lichen'], 'system:time_start': 1609459200000, 'system:time_end': 1640991600000, 'Map_class_palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4', 'f0f0f0', '0064c8', '0096a0', '00cf75', 'fae6a0'], 'Map_class_values': [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100], 'system:asset_size': 109661138988, 'system:index': '2021'}}


## Define Landcover Classes of Interest

Value	Color	Description
10	#006400	Tree cover
20	#ffbb22	Shrubland
30	#ffff4c	Grassland
40	#f096ff	Cropland
50	#fa0000	Built-up
60	#b4b4b4	Bare / sparse vegetation
70	#f0f0f0	Snow and ice
80	#0064c8	Permanent water bodies
90	#0096a0	Herbaceous wetland
95	#00cf75	Mangroves
100	#fae6a0	Moss and lichen

#### Values to Extract
- 10: Tree cover --> this should be dark, but not as dark as 60. Will include forests.
- 20: Shrubland --> this should be darker than 30, but not as dark as 10. Will include savannahs.
- 30: Grassland --> this should be the brightest of the vegetated areas. Will include grasslands.
- 40: Cropland --> this should be brighter than 10, but not as bright as 30. Will include farmlands.
- 50: Built-up --> this should be bright, brighter than anything else hopefully. Will include bright cities.
- 60: Bare / sparse vegetation --> this should mostly be desert, hopefully the darkest.

#### Values to Skip
- 70: Snow and ice --> there isn't much at all int he continent
- 80: Permanent water bodies --> I should get this from my ocean polygon
- 90: Herbaceous wetland --> also don't know how much of this there is
- 95: Mangroves --> there are some, but not in most countries

## Select Just for Dark Areas that are far away from Built Areas

In [38]:

# create a mask for the desired landcovers
built_mask = landcover_africa.eq(50)
built = landcover_africa.updateMask(built_mask)
# m.addLayer(built, {'palette': 'red'}, 'Built')

# create a 10km buffer around built areas
built_buffer = built.focal_max(10000, 'circle', 'meters')
# m.addLayer(built_buffer, {'palette': 'red'}, 'Built buffer', False)

# create a mask for the built_buffer
# need to unmask it to convert areas outside of mask from nodata to 0
rural_mask = built_buffer.eq(50).unmask(0).eq(0)
# select areas outside of the built buffer in countries of interest
rural = landcover_africa.updateMask(rural_mask)
# m.addLayer(rural, {'palette': 'brown'},  'Rural landcover', False)


## Sample Rural, Built, Ocean, Desert, and Jungle Areas
Now that we have our areas defined, let's randomly sample 700-2000 points from each landcover class. 

### Rural

In [39]:
# sample rural landcover
rural_pts = rural.sample(
    region=all_minigrid_countries_fc,
    scale=1000,
    numPixels=2000,
    seed=44,
    projection='EPSG:4326',
    geometries=True,
    dropNulls=True
)

# add a property "type" to the built_pts feature collection equal to "built"
rural_pts = rural_pts.map(lambda f: f.set('type', 'rural'))

# export rural sample to geojson
geemap.ee_to_geojson(rural_pts, 'data/dark_africa/rural_pts3.geo.json')

In [40]:
# read in the rural_pts
rural_pts = geemap.geojson_to_ee('data/dark_africa/rural_pts3.geo.json')
# add to map
m.addLayer(rural_pts, {'color': 'purple'}, 'Rural Points')
rural_pts

### Built

In [41]:
built_pts = built.sample(
    region=all_minigrid_countries_fused_fc,
    scale=1000,
    numPixels=250000, # 250k pixels to get 700 not-null values
    seed=44,
    projection='EPSG:4326',
    geometries=True,
    dropNulls=True
)

# add a property "type" to the built_pts feature collection equal to "built"
built_pts = built_pts.map(lambda f: f.set('type', 'built'))

# export built sample to geojson
geemap.ee_to_geojson(built_pts, 'data/dark_africa/built_pts2.geo.json')

In [42]:
# read in the built_pts
built_pts = geemap.geojson_to_ee('data/dark_africa/built_pts2.geo.json')
# add to map
m.addLayer(built_pts, {'color': 'red'}, 'Built Points')
built_pts

### Ocean

In [43]:
# sample ocean polygon
ocean_pts = landcover.sample(
    region=ocean_feat,
    scale=1000,
    numPixels=800,
    seed=44,
    projection='EPSG:4326',
    geometries=True,
    dropNulls=False
)

# add a property "type" to the ocean_pts feature collection equal to "ocean"
ocean_pts = ocean_pts.map(lambda f: f.set('type', 'ocean'))

# export sample to geojson
geemap.ee_to_geojson(ocean_pts, 'data/dark_africa/ocean_pts2.geo.json')

In [44]:
# read in the ocean_pts
ocean_pts = geemap.geojson_to_ee('data/dark_africa/ocean_pts2.geo.json')
# add to map
m.addLayer(ocean_pts, {'color': 'blue'}, 'Ocean Points')
ocean_pts

### Desert

In [45]:
# sample ocean polygon
desert_pts = landcover.sample(
    region=desert_feat,
    scale=1000,
    numPixels=800,
    seed=44,
    projection='EPSG:4326',
    geometries=True,
    dropNulls=True
)

# add a property "type" to the desert_pts feature collection equal to "desert"
desert_pts = desert_pts.map(lambda f: f.set('type', 'desert'))

# export sample to geojson
geemap.ee_to_geojson(desert_pts, 'data/dark_africa/desert_pts2.geo.json')

In [46]:
# read in the desert_pts
desert_pts = geemap.geojson_to_ee('data/dark_africa/desert_pts2.geo.json')
# add to map
m.addLayer(desert_pts, {'color': 'brown'}, 'Desert Points')
desert_pts

### Jungle

In [47]:
# sample ocean polygon
jungle_pts = landcover.sample(
    region=jungle_feat,
    scale=1000,
    numPixels=800,
    seed=44,
    projection='EPSG:4326',
    geometries=True,
    dropNulls=True
)

# add a property "type" to the jungle_pts feature collection equal to "jungle"
jungle_pts = jungle_pts.map(lambda f: f.set('type', 'jungle'))

# export sample to geojson
geemap.ee_to_geojson(jungle_pts, 'data/dark_africa/jungle_pts2.geo.json')

In [48]:
# read in the jungle_pts
jungle_pts = geemap.geojson_to_ee('data/dark_africa/jungle_pts2.geo.json')
# add to map
m.addLayer(jungle_pts, {'color': 'green'}, 'Rainforest Points')
jungle_pts

### Read Back in Points and Add to Map

In [49]:
# read in the sample points geojsons
built_pts = geemap.geojson_to_ee('data/dark_africa/built_pts2.geo.json')
rural_pts = geemap.geojson_to_ee('data/dark_africa/rural_pts3.geo.json')
ocean_pts = geemap.geojson_to_ee('data/dark_africa/ocean_pts2.geo.json')
desert_pts = geemap.geojson_to_ee('data/dark_africa/desert_pts2.geo.json')
jungle_pts = geemap.geojson_to_ee('data/dark_africa/jungle_pts2.geo.json')

# add the sample points to the map with appropriate colors
m.addLayer(built_pts, {'color': 'red'}, 'Built points')
m.addLayer(rural_pts, {'color': 'purple'}, 'Rural points')
m.addLayer(ocean_pts, {'color': 'blue'}, 'Ocean points')
m.addLayer(desert_pts, {'color': 'orange'}, 'Desert points')
m.addLayer(jungle_pts, {'color': 'darkgreen'}, 'Jungle points')

# print the number of points in each feature collection
print('Built points: ', built_pts.size().getInfo())
print('Rural points: ', rural_pts.size().getInfo())
print('Ocean points: ', ocean_pts.size().getInfo())
print('Desert points: ', desert_pts.size().getInfo())
print('Jungle points: ', jungle_pts.size().getInfo())


Built points:  502
Rural points:  1795
Ocean points:  800
Desert points:  800
Jungle points:  800


In [50]:
# add in mini-grid sites to map
cluber_fc = geemap.geojson_to_ee('data/cluber/cluber_sites.geojson')
m.addLayer(cluber_fc, {'color': 'yellow'}, 'Club-ER Sites')

In [51]:
# rural_trees_mask = rural.eq(10)
# rural_shrub_mark = rural.eq(20)
# rural_grass_mask = rural.eq(30)
# rural_crop_mask = rural.eq(40)
# rural_bare_mask = rural.eq(60)

# rural_trees = rural.updateMask(rural_trees_mask)
# rural_shrub = rural.updateMask(rural_shrub_mark)
# rural_grass = rural.updateMask(rural_grass_mask)
# rural_crop = rural.updateMask(rural_crop_mask)
# rural_bare = rural.updateMask(rural_bare_mask)

# note this drops any points that have been masked
# rural_trees_points = rural_rural.sample(
#     region=all_minigrid_countries_fc.geometry(),
#     scale=30, # 1000km
#     numPixels=1, # 10k points, many get dropped
#     seed=44,
#     dropNulls=True, # drop any points that have been masked
#     geometries=True
# )
# # convert into a feature collection of points
# rural_rural_fc = ee.FeatureCollection(rural_trees_points)

# save the feature collections
# geemap.ee_export_vector(rural_trees_fc, 'data/dark_africa/rural_trees_points.geojson')


## Create a 1km Buffer Around the Sampled Points

In [52]:
# read in the feature collections from geojson
built_pts_fc = geemap.geojson_to_ee('data/dark_africa/built_pts2.geo.json')
ocean_pts_fc = geemap.geojson_to_ee('data/dark_africa/ocean_pts2.geo.json')
desert_pts_fc = geemap.geojson_to_ee('data/dark_africa/desert_pts2.geo.json')
jungle_pts_fc = geemap.geojson_to_ee('data/dark_africa/jungle_pts2.geo.json')
rural_pts_fc = geemap.geojson_to_ee('data/dark_africa/rural_pts3.geo.json')

# combine the feature collections
dark_pts_fc = built_pts_fc.merge(ocean_pts_fc).merge(desert_pts_fc).merge(jungle_pts_fc).merge(rural_pts_fc)

In [53]:
# inspect the size of the fc
print('Number of points in built_pts_fc: ', built_pts_fc.size().getInfo())
print('Number of points in ocean_pts_fc: ', ocean_pts_fc.size().getInfo())
print('Number of points in desert_pts_fc: ', desert_pts_fc.size().getInfo())
print('Number of points in jungle_pts_fc: ', jungle_pts_fc.size().getInfo())
print('Number of points in rural_pts_fc: ', rural_pts_fc.size().getInfo())
# combined
print('Number of points in dark_pts_fc: ', dark_pts_fc.size().getInfo())


Number of points in built_pts_fc:  502
Number of points in ocean_pts_fc:  800
Number of points in desert_pts_fc:  800
Number of points in jungle_pts_fc:  800
Number of points in rural_pts_fc:  1795
Number of points in dark_pts_fc:  4697


In [54]:
# create a 1km buffer around the rural water points
dark_pts_buffer = dark_pts_fc.map(lambda f: f.buffer(1000))


## Get the Nightlights Values for each buffer

In [55]:
# get the image collectino of nightlight images
dataset_night = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG') \
                  .filter(ee.Filter.date('2014-01-01', '2024-01-01'))
# select the avg_rad band
nighttime_ic = dataset_night.select('avg_rad')
# stack the images into a single image
nighttime_ic_stack = nighttime_ic.toBands()
nighttime_ic_stack

In [56]:
# apply the reduceRegions for dark
nighttime_dark_fc = nighttime_ic_stack.reduceRegions(
    collection=dark_pts_buffer, 
    reducer=ee.Reducer.mean(), 
    scale=50
)


In [57]:
# get the centerpoint of each polygon feature in the feature collection
nighttime_dark_fc = nighttime_dark_fc.map(lambda f: f.set('center', f.geometry().centroid().coordinates()))

## Convert the Feature Collections to GeoDataFrames and Clean Data

In [ ]:
# convert fc to gdf
nighttime_dark_gdf = geemap.ee_to_geopandas(nighttime_dark_fc)
# nighttime_dark_gdf.head(5)

In [ ]:
# remove "_avg_rad" from the column names that contain it
nighttime_dark_gdf.columns = nighttime_dark_gdf.columns.str.replace('_avg_rad', '')
nighttime_dark_gdf.head(6)

,geometry,20140101,20140201,20140301,20140401,20140501,20140601,20140701,20140801,20140901,...,20230501,20230601,20230701,20230801,20230901,20231001,20231101,Map,center,type
0,"POLYGON ((2.33339 6.55256, 2.33086 6.55220, 2....",0.342968,0.402747,0.591590,0.627920,0.841341,0.789663,2.050187,0.862919,0.717496,...,2.482867,2.783915,2.775975,2.304453,2.980115,3.340708,3.500435,50.0,"[2.333387535907443, 6.543558933186543]",built
1,"POLYGON ((7.56328 5.72232, 7.56075 5.72196, 7....",0.585448,0.423186,0.713340,1.380553,-0.062595,0.050709,0.390967,0.471818,1.148631,...,0.000000,0.367120,0.237813,0.360027,0.008575,0.009520,0.511216,50.0,"[7.563278792531629, 5.713322064494402]",built
2,"POLYGON ((-4.33668 9.93446, -4.33923 9.93410, ...",-0.009527,0.061128,0.100562,0.055074,0.100940,-0.005170,-0.005658,0.076313,0.077994,...,0.310717,0.428970,0.334139,0.365331,0.332577,0.395312,0.411871,50.0,"[-4.336682376746656, 9.925465263755695]",built
3,"POLYGON ((35.37758 -23.30052, 35.37485 -23.300...",1.624531,1.620947,1.721038,1.545961,1.680671,1.652644,1.675032,1.506489,1.624703,...,2.602425,2.401618,2.592629,2.642773,2.214580,2.443272,2.718274,50.0,"[35.37758294573388, -23.30952150601561]",built
4,"POLYGON ((2.34888 6.55819, 2.34635 6.55783, 2....",0.291702,0.357614,0.400716,0.496250,0.387205,0.410198,1.162781,0.676260,0.449396,...,2.897713,3.043083,2.831049,2.563683,2.886312,3.186137,3.479030,50.0,"[2.348880740780288, 6.549193017854241]",built
5,"POLYGON ((39.33439 8.40063, 39.33185 8.40027, ...",1.527353,1.399204,1.441398,1.219187,0.805832,0.892052,0.674718,0.779813,0.857485,...,1.250543,1.248711,1.369335,1.153369,1.282867,1.705389,1.653763,50.0,"[39.33439164475328, 8.391627201286694]",built


In [ ]:
from shapely.geometry import Point

# convert the geometry column from a list of coordinates to the centroid of the polygon
nighttime_dark_gdf['geometry'] = nighttime_dark_gdf['center'].apply(lambda x: Point(x[0], x[1]))

# inspect head
nighttime_dark_gdf.head(6)

,geometry,20140101,20140201,20140301,20140401,20140501,20140601,20140701,20140801,20140901,...,20230501,20230601,20230701,20230801,20230901,20231001,20231101,Map,center,type
0,POINT (2.33339 6.54356),0.342968,0.402747,0.591590,0.627920,0.841341,0.789663,2.050187,0.862919,0.717496,...,2.482867,2.783915,2.775975,2.304453,2.980115,3.340708,3.500435,50.0,"[2.333387535907443, 6.543558933186543]",built
1,POINT (7.56328 5.71332),0.585448,0.423186,0.713340,1.380553,-0.062595,0.050709,0.390967,0.471818,1.148631,...,0.000000,0.367120,0.237813,0.360027,0.008575,0.009520,0.511216,50.0,"[7.563278792531629, 5.713322064494402]",built
2,POINT (-4.33668 9.92547),-0.009527,0.061128,0.100562,0.055074,0.100940,-0.005170,-0.005658,0.076313,0.077994,...,0.310717,0.428970,0.334139,0.365331,0.332577,0.395312,0.411871,50.0,"[-4.336682376746656, 9.925465263755695]",built
3,POINT (35.37758 -23.30952),1.624531,1.620947,1.721038,1.545961,1.680671,1.652644,1.675032,1.506489,1.624703,...,2.602425,2.401618,2.592629,2.642773,2.214580,2.443272,2.718274,50.0,"[35.37758294573388, -23.30952150601561]",built
4,POINT (2.34888 6.54919),0.291702,0.357614,0.400716,0.496250,0.387205,0.410198,1.162781,0.676260,0.449396,...,2.897713,3.043083,2.831049,2.563683,2.886312,3.186137,3.479030,50.0,"[2.348880740780288, 6.549193017854241]",built
5,POINT (39.33439 8.39163),1.527353,1.399204,1.441398,1.219187,0.805832,0.892052,0.674718,0.779813,0.857485,...,1.250543,1.248711,1.369335,1.153369,1.282867,1.705389,1.653763,50.0,"[39.33439164475328, 8.391627201286694]",built


In [ ]:
# drop the center column
nighttime_dark_gdf = nighttime_dark_gdf.drop(columns=['center'])

In [ ]:
# export the geodataframe to a csv
nighttime_dark_gdf.to_csv('data/dark_africa/nighttime_dark_gdf.csv', index=False)

In [64]:
# export to HTML for webpage
import os
os.system('jupyter nbconvert --to html 11_AfricaGetDarkAreas.ipynb --HTMLExporter.theme=dark')

[NbConvertApp] Converting notebook 11_AfricaGetDarkAreas.ipynb to html
[NbConvertApp] Writing 4619376 bytes to 11_AfricaGetDarkAreas.html


0

## Conclusion

We have successfully extracted nightlight brightness values for mini-grid locations as well as for randomly sampled dark areas across Sub-Saharan Africa. This data can now be used for further analysis, such as comparing the brightness levels of mini-grid locations to those of uninhabited areas, which can provide insights into the detectability of mini-grid electrification efforts in the region. See the next notebook for how to run difference-in-differences analysis on this data set.